In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, explode, split

In [3]:
spark = SparkSession.builder.appName("TwitterAnalysis").getOrCreate()

26/02/28 04:10:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [22]:
%%time
df_raw = spark.read.json("gs://big-data-lunes-20260223/war_tweets.txt")

CPU times: user 132 ms, sys: 14.1 ms, total: 147 ms
Wall time: 2min 9s


In [5]:
# Inspeccionar la estructura (verás los campos: user, entities, text, etc.)
df_raw.printSchema()

root
 |-- _type: string (nullable = true)
 |-- card: struct (nullable = true)
 |    |-- _type: string (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- thumbnailUrl: string (nullable = true)
 |    |-- title: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- cashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- content: string (nullable = true)
 |-- conversationId: long (nullable = true)
 |-- coordinates: struct (nullable = true)
 |    |-- _type: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: long (nullable = true)
 |-- inReplyToTweetId: long (nullable = true)
 |-- inReplyToUser: struct (nullable = true)
 |    |-- _type: string (nullable = true)
 |    |-- created: string (nullable = true)
 |    |-- description: 

### El formato importa
Tomemos el archivo, y vamos a guardarlo como parquet.

In [16]:
#df_raw.write.mode("overwrite").parquet("gs://big-data-lunes-20260223/war_tweets_parquet")

In [23]:
%%time
df_p = spark.read.parquet("gs://big-data-lunes-20260223/war_tweets_parquet")

CPU times: user 3.73 ms, sys: 63 µs, total: 3.79 ms
Wall time: 1.66 s


In [24]:
df_p.printSchema()

root
 |-- _type: string (nullable = true)
 |-- card: struct (nullable = true)
 |    |-- _type: string (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- thumbnailUrl: string (nullable = true)
 |    |-- title: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- cashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- content: string (nullable = true)
 |-- conversationId: long (nullable = true)
 |-- coordinates: struct (nullable = true)
 |    |-- _type: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: long (nullable = true)
 |-- inReplyToTweetId: long (nullable = true)
 |-- inReplyToUser: struct (nullable = true)
 |    |-- _type: string (nullable = true)
 |    |-- created: string (nullable = true)
 |    |-- description: 

# Inspecting and selecting

In [36]:
df_tweets = df_p.select(
    col("id").alias("tweet_id"),
    col("date").alias("created_at"),
    col("content"),
    col("user.username").alias("username"),
    col("user.displayname").alias("handle"),
    col("user.location").alias("location"))

In [37]:
df_tweets.show()

+-------------------+--------------------+--------------------+---------------+--------------------+--------------------+
|           tweet_id|          created_at|             content|       username|              handle|            location|
+-------------------+--------------------+--------------------+---------------+--------------------+--------------------+
|1506955322082091015|2022-03-24T11:25:...|@strategywoman Go...|   AdvSafetyIRL|        David Butler|    Wicklow, Ireland|
|1506955320840658947|2022-03-24T11:25:...|Journalist report...|ConsultantsFrog|Flying Frog Consu...|         Atlanta, GA|
|1506955320295317509|2022-03-24T11:25:...|@RussianEmbassy @...|      Niccomanz|                Niko|                    |
|1506955318483427333|2022-03-24T11:25:...|@globaltimesnews ...| d_serebriakova|  Daria Serebriakova|                    |
|1506955318319730688|2022-03-24T11:25:...|@Dave_nobody @IAP...|  hong386461771|            🌻虹🌈🌻|                    |
|1506955318152019973|2022-03

In [38]:
df_tweets.count()

6798932

In [47]:
#Usuarios verificados con más de 100 likes
verified = df_p.filter(
    (col("user.verified")==True) & (col("likeCount") > 1000)
).select(col("user.username").alias("username"),
    col("user.displayname").alias("handle"),
    col("user.location").alias("location"))

In [48]:
verified.show()

+---------------+--------------------+-------------------+
|       username|              handle|           location|
+---------------+--------------------+-------------------+
|KyivIndependent|The Kyiv Independent|               Kyiv|
| TelegraphWorld|Telegraph World News|             London|
|        lcvelez|Luis Carlos Vélez 🌎|      United States|
|      pmarsupia|  Principia Marsupia|               Kiev|
|KyivIndependent|The Kyiv Independent|               Kyiv|
|KyivIndependent|The Kyiv Independent|               Kyiv|
|     AFPespanol|Agence France-Presse|Montevideo, Uruguay|
| RussianEmbassy| Russian Embassy, UK|             London|
|KyivIndependent|The Kyiv Independent|               Kyiv|
| Volksverpetzer|      Volksverpetzer|                   |
|   kylegriffin1|        Kyle Griffin|    Los Angeles, CA|
| diazvillanueva|     Díaz Villanueva|             Madrid|
|KyivIndependent|The Kyiv Independent|               Kyiv|
|     ianbremmer|         ian bremmer|                   

In [51]:
#manejando arreglos
from pyspark.sql.functions import explode

df_hashtags = df_p.select("id", explode("hashtags").alias("hashtag"))
df_hashtags.show()


+-------------------+------------------+
|                 id|           hashtag|
+-------------------+------------------+
|1506955320295317509|      humanitarian|
|1506955320295317509|           Ukraine|
|1506955317476741122|           Ukraine|
|1506955317476741122|        sketchbook|
|1506955317476741122|            sketch|
|1506955317476741122|        Ukrainians|
|1506955316533121031|           MundoKW|
|1506955307519516675|               Usa|
|1506955307519516675|           Ucraina|
|1506955305241956359|GOPtheRussianParty|
|1506955305241956359|         PutinsWar|
|1506955305241956359|    PutinWarCrimes|
|1506955304520630280|  StandWithUkraine|
|1506955303639736328|           ukrayna|
|1506955303639736328|             putin|
|1506955299248349184|              EEUU|
|1506955299248349184|           Ucrania|
|1506955296945680401|  StopWarInUkraine|
|1506955295221764097|    PutinWarCrimes|
|1506955292126371841|         SmartNews|
+-------------------+------------------+
only showing top

In [52]:
%%time
df_hashtags.groupBy("hashtag").count().orderBy(col("count").desc()).show(10)

+-----------------+------+
|          hashtag| count|
+-----------------+------+
|          Ukraine|460465|
|            Putin|141821|
|           Russia|122433|
|          Ucrania| 53731|
|          ukraine| 47825|
| UkraineRussiaWar| 45956|
| StandWithUkraine| 42167|
|       UkraineWar| 30004|
|          Russian| 27460|
|UkraineRussianWar| 26143|
+-----------------+------+
only showing top 10 rows



In [53]:
%%time
df_hashtags.groupBy("hashtag").count().show(10)

+--------------------+-----+
|             hashtag|count|
+--------------------+-----+
|              poetry|  381|
|             ukraina|  769|
|                hope|  401|
|              Lavrov| 1340|
|           EnDirecto|  186|
|              Attali|   15|
|verschwörungstheorie|    2|
|                 art|  811|
|           stopputin|  948|
|                 PMI|   24|
+--------------------+-----+
only showing top 10 rows



In [56]:
#Trabajo con Timestamps (Casting)
#La columna date viene como string. Vamos a convertirla a formato fecha para extraer el día de la semana.

from pyspark.sql.functions import to_timestamp, date_format

df_with_time = df_p.withColumn("timestamp", to_timestamp("date")) \
                 .withColumn("day_of_week", date_format(col("timestamp"), "EEEE"))

df_with_time.groupBy("day_of_week").count().show()

+-----------+-------+
|day_of_week|  count|
+-----------+-------+
|    Tuesday| 586523|
|     Friday|1289552|
|   Thursday|1363105|
|     Monday| 632383|
|  Wednesday| 701894|
|   Saturday|1095596|
|     Sunday|1129879|
+-----------+-------+



In [60]:
#Extraer latitud y longitud del objeto coordinates solo para los tweets que tienen GPS activado.

df_geo = df_p.filter(col("coordinates").isNotNull()) \
           .select(
               "id",
               "content",
               col("user.username").alias("username"),
               col("coordinates.latitude").alias("lat"),
               col("coordinates.longitude").alias("lon")
           )
df_geo.show(5)

+-------------------+--------------------+---------------+-----------------+-----------------+
|                 id|             content|       username|              lat|              lon|
+-------------------+--------------------+---------------+-----------------+-----------------+
|1506955317476741122|I will finish thi...|    nystyamanga|        44.386383|       22.1357201|
|1506955296945680401|"All we need here...|      javigarp_|       45.0396596|       29.3945312|
|1506955281359585282|Egypt asks for IM...|Sambryanbuabeng|          5.51713|       -0.3470252|
|1506955271100317697|@ActualidadRT Paz...|     leorovayo1|-1.72751399865014|-79.2911380019509|
|1506955202854895620|Alassane Dramane ...|NDIAYEE56561013|       12.3067185|      -17.5302485|
+-------------------+--------------------+---------------+-----------------+-----------------+
only showing top 5 rows



In [62]:
#Agregaciones complejas por Usuario
#Calcular el promedio de likes y el total de retweets por cada usuario único.
df_user_stats = df_p.groupBy("user.username") \
                  .agg(
                      {"likeCount": "avg", "retweetCount": "sum", "id": "count"}
                  ) \
                  .withColumnRenamed("count(id)", "total_tweets")

df_user_stats.filter("total_tweets > 5").show()

+---------------+--------------------+------------+-----------------+
|       username|      avg(likeCount)|total_tweets|sum(retweetCount)|
+---------------+--------------------+------------+-----------------+
| EsperanzaRicoL|  0.7272727272727273|          22|                7|
|    LeMooncrash|  1.4761904761904763|          21|                0|
|      BattoAjax| 0.36363636363636365|          55|                1|
|     TarasKuzio|    53.3875968992248|         129|             2190|
|       geo_maps|                 0.0|           8|                0|
|       info24bd|                0.22|          50|                2|
|       LEXPRESS|   6.463157894736842|         190|              732|
|Stoppelhopser67|  0.3333333333333333|           6|                0|
|    lordrodboro|                 0.0|          15|                0|
|     DEZMinutos|  0.2222222222222222|           9|                0|
|  CelarekRobert|                 0.0|          11|                0|
|       ROYA__Ro|   

In [66]:
#Encontrar el tweet más "exitoso" (más likes) de cada usuario.
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

windowSpec = Window.partitionBy("user.username").orderBy(col("likeCount").desc())

df_top_tweets = df_p.withColumn("rank", rank().over(windowSpec)) \
                  .filter(col("rank") == 1) \
                  .select("user.username", "likeCount", "content")

df_top_tweets.show(10)

+---------------+---------+--------------------+
|       username|likeCount|             content|
+---------------+---------+--------------------+
|00007_Jamesbond|       26|Thukra kar mera p...|
|     0000gokkai|        0|@Terry53502902 @R...|
|     0000gokkai|        0|@PeapeachesPea @G...|
|         00013e|        0|@OlgaNYC1211 Inde...|
|        0011Kay|        0|@MailOnline That ...|
|      001Carlao|        0|@Felipe00571151 @...|
|      001Carlao|        0|@Felipe00571151 @...|
|   001Christian|        1|It is our old fri...|
|   001Christian|        1|This will turn up...|
|   001Christian|        1|@ivanastradner Tr...|
+---------------+---------+--------------------+
only showing top 10 rows



In [ ]:
# 'transform' permite aplicar una función a cada elemento de un array sin explotarlo
from pyspark.sql.functions import expr

df_mentions = df.withColumn("mentioned_usernames", 
                           expr("transform(mentionedUsers, x -> x.username)"))

df_mentions.select("content", "mentioned_usernames").show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import col, lower, split, explode, trim

# 1. Cargar como DataFrame
df_tweets = spark.read.text(PATH_TWEETS).withColumnRenamed("value", "text")

# 2. Transformaciones en cadena (Pipeline)
# Limpiamos espacios, pasamos a minúsculas y separamos por palabras
df_words = df_tweets.withColumn("clean_text", lower(trim(col("text")))) \
                    .withColumn("word", explode(split(col("clean_text"), " "))) \
                    .filter(col("word") != "") # Eliminar espacios vacíos

df_words.cache() # Guardamos en RAM de los Workers para velocidad

In [ ]:
# Realizamos un conteo de palabras
word_counts = df_words.groupBy("word").count().orderBy(col("count").desc())

# ACCIÓN: Esto disparará el Job en Dataproc
word_counts.show(10)

# Ver el plan de ejecución
# Pide a los alumnos que busquen "Exchange" (Shuffle) en el plan
word_counts.explain()

In [ ]:
keywords = ["war", "peace", "conflict", "world"]
df_filtered = df_words.filter(col("word").isin(keywords))

# Agrupar y mostrar
df_filtered.groupBy("word").count().show()

# Bloque 2, SparkSQL

In [67]:
df_p.createOrReplaceTempView("v_tweets")

# Verificar que Spark reconoce la tabla
spark.catalog.listTables()

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


[Table(name='v_tweets', database=None, description=None, tableType='TEMPORARY', isTemporary=True)]

In [68]:
spark.sql("""
    SELECT id, content, user.username, user.followersCount, user.location
    FROM v_tweets
    WHERE user.followersCount > 10000
    ORDER BY user.followersCount DESC
""").show(5)

+-------------------+--------------------+-----------+--------------+--------------+
|                 id|             content|   username|followersCount|      location|
+-------------------+--------------------+-----------+--------------+--------------+
|1507085038940262408|Today @POTUS anno...|BarackObama|     131464763|Washington, DC|
|1504433478141714434|Biden will speak ...|     cnnbrk|      62808206|    Everywhere|
|1504545212231831553|The House has pas...|     cnnbrk|      62807297|    Everywhere|
|1504901237824761859|President Biden u...|     cnnbrk|      62805034|    Everywhere|
|1505540257252622337|Ukrainian Preside...|     cnnbrk|      62803282|    Everywhere|
+-------------------+--------------------+-----------+--------------+--------------+
only showing top 5 rows



In [69]:
#Buscar tweets que hablen específicamente de un tema sin necesidad de "explotar" la lista.
spark.sql("""
    SELECT content, hashtags
    FROM v_tweets
    WHERE array_contains(hashtags, 'Ukraine') 
       OR array_contains(hashtags, 'Peace')
""").show(5)

+--------------------+--------------------+
|             content|            hashtags|
+--------------------+--------------------+
|@RussianEmbassy @...|[humanitarian, Uk...|
|I will finish thi...|[Ukraine, sketchb...|
|There we have it ...|[Ukraine, StopRus...|
|NEW (via WH press...|[NATOsummit, Brus...|
|Aktuelle Befragun...|[KMU, Deutschland...|
+--------------------+--------------------+
only showing top 5 rows



In [71]:
#Para contar la frecuencia de cada hashtag, debemos convertir la lista en filas individuales.
spark.sql("""
    SELECT h.hashtag, COUNT(*) as total
    FROM v_tweets as h
    LATERAL VIEW explode(hashtags) t AS hashtag
    GROUP BY h.hashtag
    ORDER BY total DESC
    LIMIT 10
""").show()

AnalysisException: Column 'h.hashtag' does not exist. Did you mean one of the following? [t.hashtag, h.hashtags, h.cashtags, h.date, h.lang, h.card, h.media, h.user, h._type, h.content, h.id, h.place, h.source, h.url, h.outlinks, h.likeCount, h.sourceUrl, h.coordinates, h.quoteCount, h.replyCount, h.sourceLabel, h.tcooutlinks, h.quotedTweet, h.retweetCount, h.conversationId, h.inReplyToUser, h.retweetedTweet, h.mentionedUsers, h.renderedContent, h.inReplyToTweetId]; line 2 pos 11;
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Sort ['total DESC NULLS LAST], true
      +- 'Aggregate ['h.hashtag], ['h.hashtag, count(1) AS total#1458L]
         +- Generate explode(hashtags#447), false, t, [hashtag#1460]
            +- SubqueryAlias h
               +- SubqueryAlias v_tweets
                  +- View (`v_tweets`, [_type#440,card#441,cashtags#442,content#443,conversationId#444L,coordinates#445,date#446,hashtags#447,id#448L,inReplyToTweetId#449L,inReplyToUser#450,lang#451,likeCount#452L,media#453,

In [72]:
#¿Desde qué plataforma (iPhone, Android, Web) se generan más Retweets en promedio?
spark.sql("""
    SELECT sourceLabel, 
           COUNT(*) as num_tweets, 
           ROUND(AVG(retweetCount), 2) as avg_retweets
    FROM v_tweets
    GROUP BY sourceLabel
    HAVING num_tweets > 10
    ORDER BY avg_retweets DESC
""").show()

+--------------------+----------+------------+
|         sourceLabel|num_tweets|avg_retweets|
+--------------------+----------+------------+
|     The White House|        42|      4401.5|
|Zero Hedge Publis...|        79|      132.86|
|       Gain Platform|        34|      124.56|
|           Typefully|       290|      114.53|
|SnapStream TV Search|       405|       85.45|
|           Liveuamap|       132|       80.28|
|Khoros Publishing...|        67|       74.15|
|         Twitter Ads|       152|       72.55|
|        palmerreport|        41|        72.2|
|cnews_twitter_pro...|        54|       63.33|
|          ACI Prensa|        41|       57.46|
|SkyNews Alerts - ...|        91|       50.02|
|         slashdotbot|        13|       45.85|
|            Sprinklr|      1080|       45.81|
|         TMZ Twitter|        12|       45.08|
|Blackbird Video P...|        11|       44.82|
|Twitter Media Studio|      9543|       42.68|
|         AlbertoNews|       228|       42.67|
|          Ta

In [73]:
#La columna date es un String. Vamos a convertirla a Timestamp para extraer la hora del día.
spark.sql("""
    SELECT content, 
           hour(cast(date as timestamp)) as hour_of_day,
           dayofweek(cast(date as timestamp)) as day_num
    FROM v_tweets
    LIMIT 10
""").show()

+--------------------+-----------+-------+
|             content|hour_of_day|day_num|
+--------------------+-----------+-------+
|En minutos estare...|         17|      4|
|@BakersMoney20 @P...|         17|      4|
|@RaquelLpez5 @Tax...|         17|      4|
|More aid passed o...|         17|      4|
|@kettledwhistler ...|         17|      4|
|The issue of #NAT...|         17|      4|
|Putin’in 195 bin ...|         17|      4|
|y la guerra contr...|         17|      4|
|@DirectedByMM @wa...|         17|      4|
|Noted Post-Soviet...|         17|      4|
+--------------------+-----------+-------+



In [74]:
#Subconsultas y CTEs (Common Table Expressions)
#Es ideal para limpiar datos antes de procesarlos. Aquí filtramos nulos y luego contamos por país.
spark.sql("""
    WITH clean_data AS (
        SELECT id, content, place.country as country
        FROM v_tweets
        WHERE place.country IS NOT NULL
    )
    SELECT country, COUNT(*) as tweets_count
    FROM clean_data
    GROUP BY country
    ORDER BY tweets_count DESC
""").show()

+--------------+------------+
|       country|tweets_count|
+--------------+------------+
| United States|       27156|
|United Kingdom|        8730|
|        Brazil|        7349|
|         Spain|        7331|
|         Italy|        6503|
|       Germany|        4711|
|     Argentina|        4350|
|       Ukraine|        4310|
|        Canada|        3050|
|        France|        2851|
|         India|        2041|
|        Mexico|        1867|
|        Poland|        1677|
|       Ireland|        1407|
|     Venezuela|        1403|
|      Colombia|        1351|
|     Australia|        1332|
|         Chile|        1113|
|   Switzerland|         988|
|      Portugal|         926|
+--------------+------------+
only showing top 20 rows



In [75]:
#Análisis de Menciones (Arrays de Structs)
#La columna mentionedUsers es un array de objetos. Vamos a extraer el displayname de los usuarios mencionados.
spark.sql("""
    SELECT content, 
           transform(mentionedUsers, x -> x.displayname) as people_mentioned
    FROM v_tweets
    WHERE size(mentionedUsers) > 0
""").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|content                                                                                                                                                                                                                                                                                                                                                                                  

In [79]:
#Encontrar el tweet más popular (por likeCount) de cada usuario único.

spark.sql("""
    SELECT username, content, likeCount
    FROM (
        SELECT user.username, content, likeCount,
               RANK() OVER (PARTITION BY user.username ORDER BY likeCount DESC) as rnk
        FROM v_tweets
    )
    WHERE rnk = 1
    and likeCount > 0
""").show(10)

+---------------+--------------------+---------+
|       username|             content|likeCount|
+---------------+--------------------+---------+
|00007_Jamesbond|Thukra kar mera p...|       26|
|   001Christian|This will turn up...|        1|
|   001Christian|It is our old fri...|        1|
|   001Christian|@ivanastradner Tr...|        1|
|       001_exia|@TaiLung2X @agued...|        2|
|      0028_juan|@Liliana07013155 ...|        1|
|   007Italiener|Gut so herr putin...|        1|
|       007LadyP|How can you not f...|        2|
|00CarlosCabrera|@mariaelisasmith ...|        5|
|  00O0O0O0OO187|ya entonces quien...|        2|
+---------------+--------------------+---------+
only showing top 10 rows



In [77]:
#El esquema tiene quotedTweet, que es un tweet completo dentro de otro. Vamos a ver quiénes citan a quién.
spark.sql("""
    SELECT user.username as author,
           content as original_comment,
           quotedTweet.user.username as quoted_author,
           quotedTweet.content as quoted_content
    FROM v_tweets
    WHERE quotedTweet IS NOT NULL    
""").show()

+--------------+--------------------+-------------+--------------------+
|        author|    original_comment|quoted_author|      quoted_content|
+--------------+--------------------+-------------+--------------------+
|   nystyamanga|I will finish thi...|  nystyamanga|Working on the sk...|
|    LAMenudero|It’s as if Ukrain...|ukraine_world|1 month, #Ukraine...|
|     dassadguy|biden doing every...|      NBCNews|President Biden a...|
|GomesJaccottet|Puft! Agora o Put...|       fambux|Pronto,agora acab...|
|    leithfadel|I'm still skeptic...|  NPA_English|Hundreds of veter...|
+--------------+--------------------+-------------+--------------------+



# Bloque 3, primer aprox de ML

In [82]:
from pyspark.sql.functions import when, col

#Vamos a definir que un tweet es "Popular" (1) si tiene más de 100 likes, y "No Popular" (0) si tiene menos.


# Creamos nuestro dataset de entrenamiento
ml_df = df_p.select(
    "content", 
    "lang", 
    col("user.verified").cast("string").alias("verified"), 
    "likeCount"
).withColumn("label", when(col("likeCount") > 100, 1).otherwise(0))

# Limpieza rápida de nulos
ml_df = ml_df.na.fill({"content": "", "lang": "en", "verified": "false"})

El código utiliza un Pipeline, que es una herramienta de Spark para organizar el flujo de datos. En lugar de aplicar cada transformación una por una, las metes en una "caja" que se ejecuta en orden.

Tokenizer (Tokenizer): Divide el texto (el tweet) en palabras individuales (tokens). Spark no entiende oraciones, entiende listas de palabras.

StopWordsRemover (StopWordsRemover): Elimina palabras que no aportan valor semántico (como "el", "de", "y", "un"). Esto reduce el "ruido" en los datos.

HashingTF (HashingTF): Convierte la lista de palabras en un vector de números. El algoritmo de ML solo entiende matemáticas, por lo que transformamos el texto en frecuencias de términos.

StringIndexer (StringIndexer): Es como un diccionario. Convierte categorías (como el idioma "es", "en") en números (0, 1, 2).

VectorAssembler (VectorAssembler): Es el paso más importante. Spark ML requiere que todas las variables (texto procesado + idioma + verificado) se junten en una sola columna tipo vector llamada features.

LogisticRegression (LogisticRegression): El "cerebro". Aprende la relación entre las features y la label (Popular/No Popular).

In [83]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, StringIndexer, VectorAssembler

# 1. Procesamiento de Texto (Content)
tokenizer = Tokenizer(inputCol="content", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="text_features", numFeatures=1000)

# 2. Procesamiento de Categorías (Lang y Verified)
index_lang = StringIndexer(inputCol="lang", outputCol="lang_index", handleInvalid="keep")
index_ver = StringIndexer(inputCol="verified", outputCol="ver_index")

# 3. Ensamblador de Vectores (Une todo en una columna 'features')
assembler = VectorAssembler(
    inputCols=["text_features", "lang_index", "ver_index"], 
    outputCol="features"
)

In [84]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Definimos el algoritmo
lr = LogisticRegression(maxIter=10, regParam=0.01, featuresCol="features", labelCol="label")

# Creamos el Pipeline (esto ordena todos los pasos anteriores)
pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, index_lang, index_ver, assembler, lr])

In [85]:
%%time
# División 80% entrenamiento, 20% prueba
train_data, test_data = ml_df.randomSplit([0.8, 0.2], seed=42)

# Entrenar el modelo (aquí es donde Spark usa los Workers al máximo)
model = pipeline.fit(train_data)

# Hacer predicciones
predictions = model.transform(test_data)

# Ver resultados
predictions.select("content", "probability", "prediction").show(5)

+--------------------+--------------------+----------+
|             content|         probability|prediction|
+--------------------+--------------------+----------+
|" .. I believe th...|[0.98845728551492...|       0.0|
|" Joe Biden veut ...|[0.97555683649456...|       0.0|
|" La meilleure fa...|[0.99187177346051...|       0.0|
|"""""""Nella situ...|[0.98783318058768...|       0.0|
|"#Bloomberg donos...|[0.99314643081646...|       0.0|
+--------------------+--------------------+----------+
only showing top 5 rows

CPU times: user 257 ms, sys: 69.8 ms, total: 327 ms
Wall time: 6min 43s


In [86]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="label")
accuracy = evaluator.evaluate(predictions)

print(f"Precisión del modelo (ROC): {round(accuracy * 100, 2)}%")

Precisión del modelo (ROC): 75.63%


In [87]:
#guardando el modelo
model_path = "gs://big-data-lunes-20260223/models/tweet_popularity_v1"
model.save(model_path)
print(f"Modelo guardado exitosamente en: {model_path}")

Modelo guardado exitosamente en: gs://big-data-lunes-20260223/models/tweet_popularity_v1


In [88]:
from pyspark.ml import PipelineModel

# 1. Cargar el modelo desde GCS
# Importante: Usamos PipelineModel porque guardamos el Pipeline completo (con el Tokenizer, etc.)
loaded_model = PipelineModel.load("gs://big-data-lunes-20260223/models/tweet_popularity_v1")

# 2. Crear datos nuevos (Tweets que nunca ha visto el modelo)
nuevos_tweets = spark.createDataFrame([
    ("Increíble victoria en el frente, la paz está cerca! #war #peace", "es", "true"),
    ("Just had a coffee and looking at the rain.", "en", "false")
], ["content", "lang", "verified"])

# 3. Hacer la predicción
# El modelo cargado ya sabe que tiene que tokenizar, limpiar y ensamblar antes de predecir
predicciones_nuevas = loaded_model.transform(nuevos_tweets)

# 4. Mostrar resultados
# 1 = Probablemente será popular, 0 = No será popular
predicciones_nuevas.select("content", "probability", "prediction").show(truncate=False)

+---------------------------------------------------------------+-----------------------------------------+----------+
|content                                                        |probability                              |prediction|
+---------------------------------------------------------------+-----------------------------------------+----------+
|Increíble victoria en el frente, la paz está cerca! #war #peace|[0.9356587844462322,0.06434121555376782] |0.0       |
|Just had a coffee and looking at the rain.                     |[0.9932905513200192,0.006709448679980801]|0.0       |
+---------------------------------------------------------------+-----------------------------------------+----------+



#Pregunta frecuente de alumno

**por qué siempre tiene los mismos stages? no importa elcódigo [Stage 58:> (0 + 6) / 59]**

Esa es una observación excelente y es una de las dudas más comunes cuando se empieza con Spark en un entorno real como Dataproc. La respuesta corta es: Spark es "perezoso" (Lazy) y reutiliza el plan de ejecución.

Aquí te explico las 3 razones técnicas de por qué ves siempre el mismo número de stages (o números muy altos):

1. La Evaluación Perezosa (Lazy Evaluation)
Spark no ejecuta nada hasta que llamas a una Acción (como .show(), .count() o .save()). Si en tu notebook corres una celda que hace un filtro, luego otra que hace un join, y al final una que hace un conteo, Spark no ha hecho nada hasta esa última celda.

Cuando finalmente ejecutas el .count(), Spark mira todo el historial de transformaciones acumuladas en el DAG (Directed Acyclic Graph) y construye los stages necesarios para llegar al resultado final desde el principio. Por eso, aunque cambies una pequeña línea, el "camino" (los stages) suele ser el mismo.

2. El Linaje de los Datos (Lineage)
A diferencia de SQL tradicional en una base de datos, Spark no "guarda" los estados intermedios a menos que tú se lo pidas. Cada vez que pides un resultado, Spark dice: "Ok, para darte esto, primero tengo que leer el JSON (Stage 1), luego filtrar (Stage 2), luego hacer el Shuffle para el Join (Stage 3)...".

Si ves que siempre llega al Stage 58, es porque Spark está rastreando una cadena de transformaciones que ya es bastante larga en tu sesión de Jupyter.

3. El Contador Global de la Sesión
El número que ves (Stage 58) es un ID incremental de tu SparkSession. No significa que ese comando específico tenga 58 pasos, sino que es el quincuagésimo octavo stage que se intenta ejecutar desde que encendiste el cluster.

Si reinicias el Kernel de Jupyter, verás que empieza otra vez desde Stage 0.

El (0 + 6) / 59 que ves significa: "Voy a ejecutar 59 tareas en total para este stage, y ahora mismo hay 6 corriendo en paralelo en tus 2 workers".

**por qué el mismo número de tareas?**

1. En la fase de Lectura (Input Split)Cuando lees un archivo de Google Cloud Storage (como tu JSON o Parquet), Spark divide el archivo en particiones lógicas.Por defecto, en GCS/HDFS, el tamaño de un "split" suele ser de 128 MB.Cálculo: $\text{Número de Tareas} = \frac{\text{Tamaño Total del Archivo}}{\text{128 MB}}$Ejemplo: Si tu archivo bank_transactions.csv pesa 5 GB ($5120\text{ MB}$), Spark creará aproximadamente $5120 / 128 = \mathbf{40}$ tareas iniciales. Cada tarea procesará un fragmento de 128 MB en un core de tus workers n2-standard-2

2. En la fase de Shuffle (Transformaciones de "Ancho")
Cuando haces un groupBy, join o distinct, los datos tienen que moverse entre los nodos (esto se llama Shuffle). Aquí es donde el número de tareas suele cambiar a un número fijo por configuración.

Spark SQL Shuffle Partitions: Por defecto, Spark está configurado para crear 200 tareas después de un shuffle.

Si ves que después de un groupBy el número de tareas salta a 200, es por esta propiedad: spark.sql.shuffle.partitions.

Tip para tu clase: Como tu cluster es pequeño (2 workers con 2 cores cada uno = 4 slots), tener 200 tareas es demasiado (mucho tiempo perdido coordinando). Puedes bajarlo para que sea más eficiente:
spark.conf.set("spark.sql.shuffle.partitions", "8") (El doble de tus cores totales es una buena regla).

3. El número que viste: (0 + 6) / 59
Desglosemos lo que significa ese indicador de progreso que mencionaste:

59 (Total de Tareas): Es el número de piezas en las que Spark dividió este "Stage" específico.

6 (Tareas Activas): Son las tareas que se están ejecutando en este preciso segundo. Como tienes 2 workers n2-standard-2 (2 CPUs cada uno) y 1 master n4-standard-2, Spark está usando los slots disponibles para procesar 6 fragmentos a la vez.

0 (Tareas Completadas): Cuántas han terminado con éxito.

In [89]:
spark.stop()